# 07 · More Data Structures: `heapq`, `bisect`, `array`, `graphlib` & Building Your Own

**Module 03 – Data Structures** · Estimated time: 50 min

> ❗ Not in either source. These standard-library tools solve common problems (top-N, priority queues, sorted inserts, scheduling) efficiently, and appear constantly in coding interviews.

### 🎯 What you will learn
- **Heaps / priority queues** with `heapq` (min-heap, 🆕 max-heap functions in 3.14, top-N, merging)
- **Binary search** and sorted inserts with `bisect`
- Compact numeric storage with `array` and `memoryview`
- **Topological sorting** (dependency order) with `graphlib`
- Implementing classic structures yourself: linked list, stack, queue, binary search tree, graph (BFS/DFS), LRU cache
- Big-O summary of Python's structures

## 1. Heaps with `heapq`

A **heap** keeps the **smallest** item at index 0. `push` and `pop` are O(log n); peeking is O(1). Perfect for **priority queues** and **top-N**.

In [1]:
import heapq

nums = [9, 4, 7, 1, 8, 2]
heapq.heapify(nums)                  # O(n), in place
print(nums, "→ smallest:", nums[0])
heapq.heappush(nums, 0)
print([heapq.heappop(nums) for _ in range(len(nums))])      # pops in ascending order

[1, 4, 2, 9, 8, 7] → smallest: 1
[0, 1, 2, 4, 7, 8, 9]


### Priority queue with tuples `(priority, counter, item)`

In [2]:
import itertools

tasks, counter = [], itertools.count()
def add_task(priority, name):
    heapq.heappush(tasks, (priority, next(counter), name))     # counter breaks ties & keeps FIFO order

add_task(3, "write docs")
add_task(1, "fix production bug")
add_task(2, "review PR")
add_task(1, "reply to CEO")
while tasks:
    priority, _, name = heapq.heappop(tasks)
    print(priority, name)

1 fix production bug
1 reply to CEO
2 review PR
3 write docs


### Top-N and merging sorted streams

In [3]:
scores = {"Asha": 91, "Ravi": 78, "Meena": 99, "John": 85, "Lee": 66}
print(heapq.nlargest(3, scores.items(), key=lambda kv: kv[1]))
print(heapq.nsmallest(2, [5, 1, 8, 3, 9]))
print(list(heapq.merge([1, 4, 9], [2, 3, 10], [0, 5])))        # lazily merge already-sorted inputs

[('Meena', 99), ('Asha', 91), ('John', 85)]
[1, 3]
[0, 1, 2, 3, 4, 5, 9, 10]


### 🆕 Max-heaps (Python 3.14)

Before 3.14 you had to push negated values (`-x`). Now there are dedicated functions:

In [4]:
h = [3, 1, 4, 1, 5, 9]
heapq.heapify_max(h)
heapq.heappush_max(h, 7)
print(h[0], [heapq.heappop_max(h) for _ in range(3)])

9 [9, 7, 5]


## 2. Binary search with `bisect`

On a **sorted** list, `bisect` finds positions in O(log n):

In [5]:
import bisect

grades_cutoffs = [60, 70, 80, 90]
letters = "FDCBA"
def grade(score):
    return letters[bisect.bisect_right(grades_cutoffs, score)]
print([grade(s) for s in (45, 60, 75, 89, 90, 100)])

sorted_list = [10, 20, 30, 40]
bisect.insort(sorted_list, 25)            # insert keeping the list sorted
print(sorted_list)

def contains(sorted_seq, x):
    i = bisect.bisect_left(sorted_seq, x)
    return i < len(sorted_seq) and sorted_seq[i] == x
print(contains(sorted_list, 30), contains(sorted_list, 31))

people = [("Ravi", 25), ("Asha", 31), ("Meena", 40)]          # sorted by age
print(bisect.bisect_left(people, 30, key=lambda p: p[1]))      # key= since 3.10

['F', 'D', 'C', 'B', 'A', 'A']
[10, 20, 25, 30, 40]
True False
1


## 3. `array` and `memoryview`: compact numeric data

A `list` of ints stores pointers to full int objects; `array.array` stores raw C numbers.

In [6]:
import array, sys

as_list = list(range(100_000))
as_array = array.array("i", range(100_000))          # "i" = signed C int (4 bytes)
print(sys.getsizeof(as_list) + sum(sys.getsizeof(x) for x in as_list), "bytes (list + int objects)")
print(sys.getsizeof(as_array), "bytes (array)")
as_array.append(5)
print(as_array[:5], as_array.typecode, as_array.itemsize)

view = memoryview(as_array)[10:15]          # zero-copy slice
print(view.tolist())

3600056 bytes (list + int objects)
408360 bytes (array)
array('i', [0, 1, 2, 3, 4]) i 4
[10, 11, 12, 13, 14]


For serious numeric work use **NumPy** (Module 10 · 09). Binary data, `bytes` and `memoryview` are covered in Module 11 · 05.

## 4. Dependency ordering with `graphlib`

In [7]:
from graphlib import TopologicalSorter, CycleError

# "task: {tasks it depends on}"
build = {
    "deploy": {"test", "build"},
    "test": {"build"},
    "build": {"install_deps"},
    "docs": {"install_deps"},
    "install_deps": set(),
}
print(list(TopologicalSorter(build).static_order()))

try:
    list(TopologicalSorter({"a": {"b"}, "b": {"a"}}).static_order())
except CycleError as e:
    print("CycleError:", e.args[1])

['install_deps', 'build', 'docs', 'test', 'deploy']
CycleError: ['a', 'b', 'a']


`TopologicalSorter` also supports **parallel** processing: `get_ready()` returns everything whose dependencies are done.

## 5. Implementing classic structures

### Linked list

In [8]:
from dataclasses import dataclass
from typing import Any

@dataclass
class ListNode:
    value: Any
    next: "ListNode | None" = None

class LinkedList:
    def __init__(self, items=()):
        self.head = None
        for item in reversed(list(items)):
            self.head = ListNode(item, self.head)
    def __iter__(self):
        node = self.head
        while node:
            yield node.value
            node = node.next
    def reverse(self):
        prev, cur = None, self.head
        while cur:
            cur.next, prev, cur = prev, cur, cur.next
        self.head = prev

ll = LinkedList([1, 2, 3, 4])
ll.reverse()
print(list(ll))

[4, 3, 2, 1]


### Binary search tree

In [9]:
class BST:
    @dataclass
    class Node:
        key: int
        left: "BST.Node | None" = None
        right: "BST.Node | None" = None

    def __init__(self):
        self.root = None
    def insert(self, key):
        def _ins(node):
            if node is None:
                return BST.Node(key)
            if key < node.key:
                node.left = _ins(node.left)
            else:
                node.right = _ins(node.right)
            return node
        self.root = _ins(self.root)
    def inorder(self, node="start"):
        node = self.root if node == "start" else node
        if node:
            yield from self.inorder(node.left)
            yield node.key
            yield from self.inorder(node.right)

t = BST()
for k in [50, 30, 70, 20, 40, 60, 80]:
    t.insert(k)
print(list(t.inorder()))

[20, 30, 40, 50, 60, 70, 80]


### Graph: BFS (shortest path) and DFS

In [10]:
from collections import deque

graph = {"A": ["B", "C"], "B": ["D"], "C": ["D", "E"], "D": ["F"], "E": ["F"], "F": []}

def bfs_shortest_path(g, start, goal):
    queue, came_from = deque([start]), {start: None}
    while queue:
        node = queue.popleft()
        if node == goal:
            path = []
            while node:
                path.append(node)
                node = came_from[node]
            return path[::-1]
        for nxt in g[node]:
            if nxt not in came_from:
                came_from[nxt] = node
                queue.append(nxt)

def dfs(g, start, seen=None):
    seen = seen if seen is not None else []
    seen.append(start)
    for nxt in g[start]:
        if nxt not in seen:
            dfs(g, nxt, seen)
    return seen

print(bfs_shortest_path(graph, "A", "F"), dfs(graph, "A"))

['A', 'B', 'D', 'F'] ['A', 'B', 'D', 'F', 'C', 'E']


### Dijkstra (weighted shortest path) with a heap

In [11]:
def dijkstra(g, start):
    dist, pq = {start: 0}, [(0, start)]
    while pq:
        d, node = heapq.heappop(pq)
        if d > dist.get(node, float("inf")):
            continue
        for nxt, w in g[node]:
            nd = d + w
            if nd < dist.get(nxt, float("inf")):
                dist[nxt] = nd
                heapq.heappush(pq, (nd, nxt))
    return dist

roads = {"Pune": [("Mumbai", 150), ("Nashik", 210)], "Mumbai": [("Nashik", 170), ("Surat", 280)],
         "Nashik": [("Surat", 250)], "Surat": []}
print(dijkstra(roads, "Pune"))

{'Pune': 0, 'Mumbai': 150, 'Nashik': 210, 'Surat': 430}


## 6. Big-O cheat-sheet

| Operation | list | deque | dict / set | heap (heapq) | sorted list + bisect |
|---|---|---|---|---|---|
| index `x[i]` | O(1) | O(n) | – | – | O(1) |
| append / push | O(1)* | O(1) | O(1) | O(log n) | O(n) insort |
| pop end / popleft | O(1) / O(n) | O(1) / O(1) | – | O(log n) (min) | – |
| search `x in` | O(n) | O(n) | **O(1)** | O(n) | **O(log n)** |
| min / max | O(n) | O(n) | O(n) | **O(1)** min | O(1) |
| sort | O(n log n) | – | – | – | already sorted |

\* amortised

## 📝 Summary
- `heapq`: priority queues, top-N, merging; 3.14 adds max-heap functions.
- `bisect`: binary search & sorted insertion (with `key=`).
- `array`/`memoryview`: compact numbers without NumPy; `graphlib`: dependency order.
- You can build linked lists, trees and graphs easily with classes, dataclasses, dicts and deques.

## 🏋️ Practice
1. Find the k most frequent words in a text using `Counter` + `heapq.nlargest`.
2. Merge k sorted lists into one sorted list with `heapq.merge`.
3. Keep a running median of a stream of numbers using two heaps.
4. Detect a cycle in a directed graph with DFS (or `graphlib`).

<details><summary>Show solution for 3</summary>

```python
import heapq

class RunningMedian:
    def __init__(self):
        self.low, self.high = [], []          # low: max-heap (negated), high: min-heap
    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))
    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

rm = RunningMedian()
for x in [5, 15, 1, 3]:
    rm.add(x)
    print(rm.median())
```
</details>